In [0]:
# Rode isso numa célula separada, sozinha, e REINICIE o Python depois (Databricks:
# menu Run > "Detach & Re-attach", ou o botão de reiniciar a sessão Python)
%pip install -q --upgrade "deltalake>=0.18"

In [0]:
%run ../config/config

In [0]:
%run ../config/squad3

In [0]:
%run ../utils/utils

In [0]:
# ==============================================================================
# UNIÃO SQUAD1 + SQUAD3 — apenas as colunas usadas no modelo de anomalia
# ==============================================================================
from deltalake import DeltaTable
import pyspark.sql.functions as F

CONTAINER_SQUAD3 = "squad3"


def get_delta_path_squad3(camada: str, tabela: str, storage_opts: dict) -> str:
    conta = storage_opts.get("account_name")
    corpo = tabela if not camada else f"{camada}/{tabela}"
    return f"abfss://{CONTAINER_SQUAD3}@{conta}.dfs.core.windows.net/{corpo}"


# CORREÇÃO: removida a primeira tentativa "sem credenciais" (ela mascarava o
# erro real, porque .load() é preguiçoso e só falha de verdade numa ação
# posterior, fora do try/except). Agora sempre usa OAuth explícito com o
# Service Principal, e força uma ação (.take(1)) dentro do try para validar
# a leitura na hora, evitando que o erro escape do bloco de proteção.
def ler_delta_squad3(camada: str, tabela: str, storage_opts: dict):
    caminho_final = get_delta_path_squad3(camada, tabela, storage_opts)
    account_name = storage_opts.get("account_name")
    client_id = storage_opts.get("client_id")
    client_secret = storage_opts.get("client_secret")
    tenant_id = storage_opts.get("tenant_id")

    df = (spark.read
        .format("delta")
        .option(f"fs.azure.account.auth.type.{account_name}.dfs.core.windows.net", "OAuth")
        .option(f"fs.azure.account.oauth.provider.type.{account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
        .option(f"fs.azure.account.oauth2.client.id.{account_name}.dfs.core.windows.net", client_id)
        .option(f"fs.azure.account.oauth2.client.secret.{account_name}.dfs.core.windows.net", client_secret)
        .option(f"fs.azure.account.oauth2.client.endpoint.{account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
        .load(caminho_final))

    try:
        df.take(1)  # força a validação AGORA, dentro do try, não depois
    except Exception as e:
        raise Exception(
            f"Erro ao ler squad3/{camada}/{tabela} com OAuth explícito. "
            f"Verifique se o Service Principal tem permissão de leitura no "
            f"container 'squad3'. Detalhe original: {e}"
        )

    return df

# ------------------------------------------------------------------
# 1. ecommerce_pedidos — só as colunas que o modelo usa
# ------------------------------------------------------------------
colunas_pedidos_modelo = ["id_pedido", "id_cliente", "dt_pedido", "valor_total", "valor_frete", "metodo_pagamento"]

df_pedidos_squad1 = (
    ler_delta("silver", "ecommerce_pedidos", STORAGE_OPTIONS)
    .select(*colunas_pedidos_modelo)
    # Prefixo para garantir que IDs não colidam entre squads (id_cliente=5 do
    # squad1 não é a mesma pessoa que id_cliente=5 do squad3)
    .withColumn("id_pedido", F.concat(F.lit("s1_"), F.col("id_pedido").cast("string")))
    .withColumn("id_cliente", F.concat(F.lit("s1_"), F.col("id_cliente").cast("string")))
    .withColumn("origem_squad", F.lit("squad1"))
)

df_pedidos_squad3 = (
    ler_delta_squad3("silver", "ecommerce_pedidos", STORAGE_OPTIONS)
    .select(*colunas_pedidos_modelo)
    .withColumn("id_pedido", F.concat(F.lit("s3_"), F.col("id_pedido").cast("string")))
    .withColumn("id_cliente", F.concat(F.lit("s3_"), F.col("id_cliente").cast("string")))
    .withColumn("origem_squad", F.lit("squad3"))
)

df_pedidos_unificado = df_pedidos_squad1.unionByName(df_pedidos_squad3)

print(f"Pedidos squad1: {df_pedidos_squad1.count()}")
print(f"Pedidos squad3: {df_pedidos_squad3.count()}")
print(f"Pedidos unificados: {df_pedidos_unificado.count()}")

# ------------------------------------------------------------------
# 2. ecommerce_itens_pedido — só as colunas que o modelo usa
# ------------------------------------------------------------------
colunas_itens_modelo = ["id_item_pedido", "id_pedido", "sku", "quantidade", "desconto_aplicado"]

df_itens_squad1 = (
    ler_delta("silver", "ecommerce_itens_pedido", STORAGE_OPTIONS)
    .select(*colunas_itens_modelo)
    .withColumn("id_item_pedido", F.concat(F.lit("s1_"), F.col("id_item_pedido").cast("string")))
    .withColumn("id_pedido", F.concat(F.lit("s1_"), F.col("id_pedido").cast("string")))
    .withColumn("sku", F.concat(F.lit("s1_"), F.col("sku")))
)

df_itens_squad3 = (
    ler_delta_squad3("silver", "ecommerce_itens_pedido", STORAGE_OPTIONS)
    .select(*colunas_itens_modelo)
    .withColumn("id_item_pedido", F.concat(F.lit("s3_"), F.col("id_item_pedido").cast("string")))
    .withColumn("id_pedido", F.concat(F.lit("s3_"), F.col("id_pedido").cast("string")))
    .withColumn("sku", F.concat(F.lit("s3_"), F.col("sku")))
)

df_itens_unificado = df_itens_squad1.unionByName(df_itens_squad3)
print(f"\nItens squad1: {df_itens_squad1.count()}")
print(f"Itens squad3: {df_itens_squad3.count()}")
print(f"Itens unificados: {df_itens_unificado.count()}")

# ------------------------------------------------------------------
# 3. ecommerce_produtos — só as colunas que o modelo usa
# ------------------------------------------------------------------
colunas_produtos_modelo = ["sku", "id_categoria"]

df_produtos_squad1 = (
    ler_delta("silver", "ecommerce_produtos", STORAGE_OPTIONS)
    .select(*colunas_produtos_modelo)
    .withColumn("sku", F.concat(F.lit("s1_"), F.col("sku")))
    .withColumn("id_categoria", F.concat(F.lit("s1_"), F.col("id_categoria").cast("string")))
)

df_produtos_squad3 = (
    ler_delta_squad3("silver", "ecommerce_produtos", STORAGE_OPTIONS)
    .select(*colunas_produtos_modelo)
    .withColumn("sku", F.concat(F.lit("s3_"), F.col("sku")))
    .withColumn("id_categoria", F.concat(F.lit("s3_"), F.col("id_categoria").cast("string")))
)

df_produtos_unificado = df_produtos_squad1.unionByName(df_produtos_squad3)
print(f"\nProdutos squad1: {df_produtos_squad1.count()}")
print(f"Produtos squad3: {df_produtos_squad3.count()}")
print(f"Produtos unificados: {df_produtos_unificado.count()}")

display(df_pedidos_unificado.limit(10))

In [0]:
# ==============================================================================
# Construir features usando os dados UNIFICADOS (squad1 + squad2)
# ==============================================================================

df_features_unificado = construir_features_pedidos(
    df_pedidos_unificado,
    df_itens_unificado,
    df_produtos_unificado
)

print(f"Total de pedidos no dataset unificado para o modelo: {df_features_unificado.count()}")
display(df_features_unificado.limit(10))

# Conferência rápida: quantos pedidos vieram de cada squad, para saber a
# proporção que cada base contribui para o treino
df_features_unificado.join(
    df_pedidos_unificado.select("id_pedido", "origem_squad"), "id_pedido"
).groupBy("origem_squad").count().show()

In [0]:
%run ../ingestion/treinamento